# Reference Checker

Hypothesis: a hallucinated reference will (a) produce a title of a paper that doesn't exist, (b) make up authors for paper titles that do exist.

This script pulls the references section out of a PDF, pulls the references, and attempts to verify each title against Semantic Scholar.
If it finds the title in Semantic Scholar, it then attempts to verify each author, according to Semantic Scholar's records, appears in the reference.

It is really hard to deal with all reference formats, but also idosycracies and casual mistakes. This script attempts to find a title by looking for a span of dictionary-recognizable words under the assumption that names do not make for long spans of dictionary-recognizable words. The algorithm is allowed to see a maximum of one non-dictionary token in a row before concluding that a span is not the title. If there is more than one span that could be a title, it will pick the longest.

This algorithm will produce a lot of false alarms where it simply fails to pull the title out of reference.

**To Use:**

Run `check_refs(filepath)`

**Notes:**
- Some PDFs that will be reviewed have line numbers. The line numbers get interjected into the middle of text spans. the `pdf_has_line_numbers=True` option will remove all numbers from references. This shouldn't matter if the pdf has line numbers or not because the algorithm should already ignore dates.
- Add words that are not in the dictionary to `CUSTOM_VOCAB`.
- Add words that you expect never to be in a paper title to `FILTER`.
- Doesn't handle names with accent marks.
- Sometimes the dictionary decides that names are all words and when there are a lot of these in a row, it will pick this over a short title.
- Sometimes reference span page breaks, in which case you get some false alarms.
- Paper headers and footers get interjected with references and create false alarms. 

# Install Packages

In [ ]:
!pip install pymupdf4llm

In [ ]:
!pip install spacy

In [ ]:
!pip install PyEnchant

# Imports

In [1]:
import pymupdf4llm
import re
import enchant
import spacy
import unidecode
import string
import requests
import time
from functools import reduce
# Load the English language model
NLP = spacy.load("en_core_web_sm")
DICTIONARY = enchant.Dict("en_US")

# Globals

In [2]:
SEMANTIC_SCHOLAR_URL = 'https://api.semanticscholar.org/graph/v1/paper/search/match?query=' 

In [3]:
CUSTOM_VOCAB = ['ai', 'xai', 'operationalizing', 'seamful', 'llm', 'llms']

In [4]:
FILTER = ['proceedings', 'conference',
          'NY', 'USA']

# Helpers

In [5]:
def tokenize(text):
    return re.findall(r"\w+|[^\w\s-]", unidecode.unidecode(text))

In [6]:
def detokenize(tokens):
    result = ''
    for token in tokens:
        if token in string.punctuation:
            result = result + token
        else:
            result = result + ' ' + token
    return result.strip()

In [7]:
def is_number(s):
    try:
        float(s)
        return True
    except ValueError:
        return False

In [8]:
def remove_numbers(text):
    return re.sub(r'\d+', '', text)

In [9]:
def does_contain(word_list, targets):
    return reduce(lambda a, b: a | b, 
                  map(lambda t: t.lower() in [w.lower() for w in word_list], 
                      targets))

In [10]:
def remove_hanging_punctuation(word_list):
    if word_list[-1] in string.punctuation:
        return word_list[0:-1]
    else:
        return word_list

# Get Reference Section 

In [11]:
def get_ref_section(md_text):
    m = re.search(r'# [0-9 ]*\*\*References\*\*([a-zA-Z0-9 \(\)\.\,\;]*)', md_text)
    if m is not None:
        after = md_text[m.span()[1]:]
        m = re.search(r'# \*\*', after)
        if m is not None:
            return after[0:m.span()[0]].strip()
        else:
            return after
    else:
        return None

# Extract Title

Each reference is on its own line. Each reference is further broken into a list of tokens (words, punctuation)

In [12]:
def extract_title(tokens, verbose=False):
    candidates = [] # Candidate titles, longest preferred
    title = [] # Current title we are building
    skip = False # We get one skip in a row
    # Iterate through tokens
    for token in tokens:
        # Part of speech tagging
        doc = NLP(token)
        if verbose:
            print(">>", token)
        # If we get a hash or star, we crash out
        # if token in ['#', '*']:
        #     return None
        # If we get certain punctuation we finish the title building
        if token in ['.', ';', '[', ']', '(', ')']:
            if verbose:
                print("PUNCT")  
                print("TITLE=", title)  
            # If title is 4 or more, then we keep it
            if len(title) > 3:
                if verbose:
                    print("CANDIDATE FOUND")
                candidates.append(title)
                title = []
                skip = False
            # If title is less than 4 we throw it out
            else:
                if verbose:
                    print("NOT A CANDIDATE")
                title = []
                skip = False
        # We cannot start a title with , or and or :
        elif (token == ',' or token == 'and' or token == ':') and len(title) == 0:
            if verbose:
                print("START WITH COMMA OR AND")
            title = []
            skip = False
        # We found something that is in the dictionaries, and is length greater than 1 (unless I or A) and is not "and"
        elif (DICTIONARY.check(token) or token.lower() in CUSTOM_VOCAB) and (len(token) > 1 or token == 'I' or token.lower() == 'a') and token.lower != 'and':
            title.append(token)
            skip = False
            if verbose:
                print("TOK")
        # Whatever remains is probably okay, but we use a skip
        elif not skip:
            skip = True
            title.append(token)
            if verbose:
                print("TOK+SKIP")
        # If we are here, we are on our second skip, give up on this
        else:
            title = []
            skip = False
            if verbose:
                print("SKIP") 
    # Now we filter out candidates
    filtered_candidates = list(filter(lambda title: not does_contain(title, FILTER), candidates)) 
    # remove orphaned punctuation
    filtered_candidates = list(map(lambda title: remove_hanging_punctuation(title),
                                   filtered_candidates))
    if len(filtered_candidates) > 0:
        sorted_candidates = sorted(filtered_candidates, key=len, reverse=True)
        return list(filter(lambda token: not is_number(token), sorted_candidates[0]))
    else:
        return None

# Access Semantic Scholar

An `author` is a json structure.

In [13]:
def get_authors_from_semantic_scholar(title):
    url = SEMANTIC_SCHOLAR_URL + title
    query_params = {"fields": "title,authors"}
    headers = {}
    response = requests.get(url, params=query_params, headers=headers)
    if response.status_code == 200:
        response_data = response.json()
        return response_data['data'][0]['authors']
    else:
        return None

In [14]:
def check_author(ref, author):
    last_name = author['name'].split()[-1]
    return last_name.lower() in ref.lower()

def check_authors(ref, authors):
    success = True
    for author in authors:
        if not check_author(ref, author):
            print("AUTHOR", author['name'], "NOT FOUND")
            success = False
    return success

# Check References

`pdf_has_line_numbers=True` will remove all numbers from each reference line.

In [15]:
def check_refs(filename, sleep=10, pdf_has_line_numbers = False):
    # Convert PDF to markdown
    print("Converting PDF to markdown...")
    md_text = pymupdf4llm.to_markdown(filename)
    # get references section, split into lines
    refs = get_ref_section(md_text).strip().replace('_', '').split('\n\n')
    # Each reference should now be a separate string in a list
    print("Checking", len(refs), "refs...")
    # Iterate through each reference line
    for n, ref in enumerate(refs):
        print(n)
        # Remove numbers if the PDF has line numbers
        if pdf_has_line_numbers:
            ref = remove_numbers(ref)
        # Get the title
        title = extract_title(tokenize(ref))
        # If title is found, check the authors
        if title is not None and len(title) > 0:
            # De-tokenize the title to get ready for Semantic Scholar search
            title = detokenize(title)
            print(title)
            # search semantic scholar and bring back a data record including authors
            authors = get_authors_from_semantic_scholar(title)
            # If authors are found then the Semantic Scholar search succeeded
            if authors is not None:
                print("FOUND in Semantic Scholar")
                # check authors
                if check_authors(ref, authors):
                    print("OK")
            # Semantic Scholar search failed
            else:
                print("NOT FOUND in Semantic Scholar")
        # No title extracted
        else:
            # Report the raw text
            print(ref)
            print('NO TITLE FOUND')
        print('\n')
        # Sleep
        time.sleep(sleep)
        

# Run Me

In [ ]:
check_refs("tests/2432.pdf", pdf_has_line_numbers = True)

Converting PDF to markdown...
=== Document parser messages ===
Using Tesseract for OCR processing.
OCR on page.number=1/2.
OCR on page.number=6/7.
OCR on page.number=7/8.
OCR on page.number=13/14.
OCR on page.number=14/15.
OCR on page.number=15/16.
OCR on page.number=16/17.

Checking 27 refs...
0
Constitutional ai: Harmlessness from ai feedback
FOUND in Semantic Scholar
AUTHOR Kamilė Lukošiūtė NOT FOUND


1
The coverage principle: How pre training enables post training
FOUND in Semantic Scholar
OK


2
Extreme parkour with legged robots
FOUND in Semantic Scholar
OK


3
Suman Pal, Pablo Samuel Castro, and Jordan Terry
NOT FOUND in Semantic Scholar


4
Deep reinforcement learning from human preferences
FOUND in Semantic Scholar
OK


5
generalizes: A comparative study of foundation model post training
FOUND in Semantic Scholar
OK


6
-  Daya Guo, Dejian Yang, Haowei Zhang, Junxiao Song, Peiyi Wang, Qihao Zhu, Runxin Xu,  Ruoyu Zhang, Shirong Ma, Xiao Bi, Xiaokang Zhang, Xingkai Yu, Yu Wu, 

# For Testing

In [ ]:
filename = "tests/2432.pdf"
md_text = pymupdf4llm.to_markdown(filename)

In [ ]:
md_text

In [ ]:
refs = get_ref_section(md_text).strip().replace('_', '').split('\n\n')
refs

In [ ]:
ref = remove_numbers(refs[4])
ref

In [ ]:
extract_title(tokenize(ref), verbose=True)